In [1]:
import logging
import sys

sys.path.append("../")

from langchain_ollama import ChatOllama

from langchain_classic.retrievers import MultiQueryRetriever

from src.loaders.document_loader import EnterpriseDocumentLoader
from src.splitters.chunker import EnterpriseChunker
from src.embeddings.embedding_manager import EmbeddingManager

# from src.vectorstores.faiss_store import EnterpriseFAISS
# from src.retrievers.hybrid_retriever import EnterpriseHybridRetriever

from vector_store.faiss_store import EnterpriseFAISS
from src.retriever.hybrid_retriever import EnterpriseHybridRetriever

c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logging.basicConfig()

logging.getLogger(
    "langchain.retrievers.multi_query"
).setLevel(logging.INFO)

In [3]:
loader = EnterpriseDocumentLoader()

documents = loader.load_directory(
    "../datasets/military"
)

chunker = EnterpriseChunker()

chunks = chunker.split_documents(documents)

In [4]:
embedding = EmbeddingManager(
    provider="ollama",
    model_name="nomic-embed-text"
)

faiss = EnterpriseFAISS(embedding)

faiss.create(chunks)

vector = faiss.retriever(k=10)

hybrid = EnterpriseHybridRetriever(
    chunks,
    vector,
    bm25_weight=0.4,
    vector_weight=0.6,
    k=10
)

In [5]:
llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [6]:
multi_query = MultiQueryRetriever.from_llm(

    retriever=hybrid.hybrid,

    llm=llm
)

In [7]:
docs = multi_query.invoke(
    "Radar Calibration Procedure"
)

print(len(docs))

20


In [8]:
for i, doc in enumerate(docs):

    print("="*80)

    print(i+1)

    print(doc.metadata)

    print()

    print(doc.page_content[:300])

1
{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 12, 'chunk_size': 207}

in  known  enemy  territory,  then  reposition     to  ambush  counter-UAV  teams  -  Overwatch  Rotation:  Coordinate  with  squad  to  maintain  100%  radar     uptime  through  sequential  UAV  deployment
2
{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 11, 'chunk_size': 489}

-  Ghost  Perk  Operatives:  Invisible  to  standard  sweeps  -  Cold-Blooded  Operatives:  No  thermal  signature  return   SECTI

In [9]:
from src.retriever.multi_query import EnterpriseMultiQuery

mq = EnterpriseMultiQuery(
    hybrid.hybrid
)

docs = mq.search(
    "Radar Calibration Procedure"
)

print(len(docs))

20


In [10]:
queries = [

    "Tank engine repair",

    "Missile inspection",

    "Radar maintenance",

    "Night vision equipment"

]

for q in queries:

    docs = mq.search(q)

    print(q)

    print(len(docs))

Tank engine repair
27
Missile inspection
33
Radar maintenance
25
Night vision equipment
25


In [11]:
queries = [

    "anticipatory bail",

    "criminal appeal",

    "contract breach",

    "consumer rights"

]

for q in queries:

    docs = mq.search(q)

    print(q)

    print(len(docs))

anticipatory bail
31
criminal appeal
29
contract breach
21
consumer rights
35
